In [ ]:
import polars as pl
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from optbinning import BinningProcess
import lightgbm as lgb
import mlflow
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri('mlruns')
mlflow.set_experiment('task1')
RUN_TAG = 'mar23'

In [ ]:
it = pl.read_csv('task1/data/IT.csv')
oot = pl.read_csv('task1/data/OOT.csv')

num_cols = [c for c in it.columns if c.startswith('NUMERICAL_')]
it = it.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))
oot = oot.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))

cutoff = it.sort('TIME_DT')['TIME_DT'].quantile(0.8)
train = it.filter(pl.col('TIME_DT') <= cutoff)
val = it.filter(pl.col('TIME_DT') > cutoff)
print(f'Train: {train.shape[0]}, Val: {val.shape[0]}')

In [ ]:
top5 = ['NUMERICAL_10', 'NUMERICAL_41', 'NUMERICAL_1', 'NUMERICAL_40', 'NUMERICAL_30']

def add_interactions(df):
    exprs = []
    for i, a in enumerate(top5):
        for b in top5[i+1:]:
            exprs.append((pl.col(a) * pl.col(b)).alias(f'{a}_x_{b}'))
            exprs.append((pl.col(a) / (pl.col(b) + 1e-8)).alias(f'{a}_div_{b}'))
            exprs.append((pl.col(a) - pl.col(b)).alias(f'{a}_minus_{b}'))
    return df.with_columns(exprs)

train_fe = add_interactions(train)
val_fe = add_interactions(val)
oot_fe = add_interactions(oot)

feat_cols = num_cols + [c for c in train_fe.columns if '_x_' in c or '_div_' in c or '_minus_' in c]
print(f'Total features: {len(feat_cols)}')

X_train = train_fe.select(feat_cols).to_pandas()
y_train = train['TARGET'].to_numpy()
X_val = val_fe.select(feat_cols).to_pandas()
y_val = val['TARGET'].to_numpy()
X_oot = oot_fe.select(feat_cols).to_pandas()

In [ ]:
lgb_train = lgb.Dataset(X_train, y_train)
lgb_val_ds = lgb.Dataset(X_val, y_val, reference=lgb_train)

configs = [
    {'num_leaves': 31, 'learning_rate': 0.05, 'min_child_samples': 100, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 0, 'reg_lambda': 0},
    {'num_leaves': 63, 'learning_rate': 0.03, 'min_child_samples': 50, 'feature_fraction': 0.7, 'bagging_fraction': 0.7, 'bagging_freq': 5, 'reg_alpha': 0.1, 'reg_lambda': 0.1},
    {'num_leaves': 15, 'learning_rate': 0.05, 'min_child_samples': 200, 'feature_fraction': 0.9, 'bagging_fraction': 0.9, 'bagging_freq': 3, 'reg_alpha': 1.0, 'reg_lambda': 1.0},
    {'num_leaves': 31, 'learning_rate': 0.02, 'min_child_samples': 100, 'feature_fraction': 0.6, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 0.5, 'reg_lambda': 0.5},
]

best_gini = 0
best_model = None
best_cfg = None

for i, cfg in enumerate(configs):
    params = {'objective': 'binary', 'metric': 'auc', 'verbosity': -1, **cfg}
    m = lgb.train(params, lgb_train, num_boost_round=1000, valid_sets=[lgb_val_ds],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    p = m.predict(X_val)
    g = 2 * roc_auc_score(y_val, p) - 1
    print(f'Config {i}: val_gini={g:.4f} (leaves={cfg["num_leaves"]}, lr={cfg["learning_rate"]})')
    if g > best_gini:
        best_gini, best_model, best_cfg = g, m, cfg

print(f'\nBest LGB: val_gini={best_gini:.4f}')

In [ ]:
bp = BinningProcess(
    variable_names=num_cols,
    min_prebin_size=0.02, min_bin_size=0.05, max_n_bins=10,
    selection_criteria={'iv': {'min': 0.02}}
)
bp.fit(train.select(num_cols).to_pandas().values, y_train)

X_tr_woe = bp.transform(train.select(num_cols).to_pandas().values, metric='woe')
X_va_woe = bp.transform(val.select(num_cols).to_pandas().values, metric='woe')

lr = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
lr.fit(X_tr_woe, y_train)
p_lr_val = lr.predict_proba(X_va_woe)[:, 1]
p_lgb_val = best_model.predict(X_val)

for w in [0.1, 0.2, 0.3, 0.4, 0.5]:
    p_ens = w * p_lr_val + (1 - w) * p_lgb_val
    g = 2 * roc_auc_score(y_val, p_ens) - 1
    print(f'Ensemble w_lr={w}: val_gini={g:.4f}')

In [ ]:
val_gini = best_gini
p_oot_final = best_model.predict(X_oot)

preds = pl.DataFrame({'ID_APPLICATION': oot['ID_APPLICATION'], 'SCORE': p_oot_final})
preds.write_csv('task1/predictions.csv')
print(f'OOT predictions saved: {preds.shape}')

with mlflow.start_run(run_name='v3_fe_tuned_lgb'):
    mlflow.log_param('model', 'LightGBM')
    mlflow.log_params(best_cfg)
    mlflow.log_param('n_features', len(feat_cols))
    mlflow.log_metric('val_gini', val_gini)
    mlflow.set_tag('task', 'task1')
    mlflow.set_tag('run_tag', RUN_TAG)
print(f'val_gini: {val_gini:.4f}')